In [3]:
import tiktoken

def get_num_tokens(text: str, model_name: str = "gpt-4.1") -> int:
    encoding = tiktoken.encoding_for_model(model_name)
    return len(encoding.encode(text))

# MedCalc

In [2]:
from datasets import load_dataset

medcalc_df = load_dataset("ncbi/MedCalc-Bench-v1.1", split="test").to_pandas()
medcalc_df = medcalc_df[['Row Number','Patient Note', 'Question']]
medcalc_df['question'] = medcalc_df['Question'].copy()#map(lambda x: x.split('?')[0].strip())
medcalc_df.head(2)

/Users/vuong/Downloads/research/repos/numeric-reasoning/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,Row Number,Patient Note,Question,question
0,1,An 87-year-old man was admitted to our hospita...,What is the patient's Creatinine Clearance usi...,What is the patient's Creatinine Clearance usi...
1,2,An 83-year-old man with a past medical history...,What is the patient's Creatinine Clearance usi...,What is the patient's Creatinine Clearance usi...


## Mean Ctx Len

In [3]:
round(medcalc_df['Patient Note'].map(lambda x: get_num_tokens(x, "gpt-4.1")).mean(),0)

np.float64(613.0)

## Mean Quesion Len

In [4]:
round(medcalc_df['question'].map(lambda x: get_num_tokens(x, "gpt-4.1")).mean(),0)

np.float64(41.0)

# Ours

In [2]:
import os
import pandas as pd

def get_full_data(dir: str):
    full_df = pd.DataFrame()
    for task in ["comparison", "summary", "retrieval", "calculation"]:
        for subtask in os.listdir(os.path.join(dir, task)):
            df = pd.read_csv(os.path.join(dir, task, subtask))
            full_df = pd.concat([full_df, df])
    full_df['context'] = full_df['question'].map(lambda x: "\n".join(x.split('\n\n')[:-1]).strip())
    full_df['question'] = full_df['question'].map(lambda x: x.split('\n\n')[-1].strip())#.split('?')[0].strip())
    return full_df

## Question length

In [8]:
data_dir = "/Users/vuong/Downloads/research/repos/numeric-reasoning/data/med_numeracy/natural"
df = get_full_data(data_dir)
df.head(2)

,question,answer,open_ended_answer,dataset_source,context
0,What is the record datetime when temperature i...,2133-05-21 05:15:00,2133-05-21 05:15:00,comparison_superlative_highest,"At 2133-05-20 23:58:00, A 77-year-old left han..."
1,What is the record datetime when temperature i...,2133-05-21 11:36:00,2133-05-21 11:36:00,comparison_superlative_lowest,"At 2133-05-20 23:58:00, A 77-year-old left han..."


In [9]:
round(df['question'].map(lambda x: get_num_tokens(x, "gpt-4.1")).mean(), 0)

np.float64(23.0)

## Structured

In [27]:
data_dir = "/Users/vuong/Downloads/research/repos/numeric-reasoning/data/med_numeracy/structured"
df = get_full_data(data_dir)
round(df['context'].map(lambda x: get_num_tokens(x, "gpt-4.1")).mean(), 0)

np.float64(745.0)

In [28]:
df['context'].iloc[0]

'{\n    chart_time: 2133-05-20 23:58:00,\n    age: 77,\n    gender: male,\n    pain: 10,\n    temperature: 36.9,\n    heart_rate: 77,\n    resp_rate: 14,\n    o2_sat: 98,\n    bp: 171/66\n},\n{\n    chart_time: 2133-05-21 05:15:00,\n    age: 77,\n    gender: male,\n    pain: NAD,\n    temperature: 37.0,\n    heart_rate: 80,\n    resp_rate: 16,\n    o2_sat: 99,\n    bp: 100/57\n},\n{\n    chart_time: 2133-05-21 09:42:00,\n    age: 77,\n    gender: male,\n    pain: uta,\n    temperature: 36.7,\n    heart_rate: 73,\n    resp_rate: 25,\n    o2_sat: 100,\n    bp: 97/52\n},\n{\n    chart_time: 2133-05-21 11:36:00,\n    age: 77,\n    gender: male,\n    pain: uta,\n    temperature: 36.6,\n    heart_rate: 65,\n    resp_rate: 30,\n    o2_sat: 100,\n    bp: 109/58\n}'

## Templated

In [24]:
data_dir = "/Users/vuong/Downloads/research/repos/numeric-reasoning/data/med_numeracy/templated"
df = get_full_data(data_dir)
round(df['context'].map(lambda x: get_num_tokens(x, "gpt-4.1")).mean(), 0)

np.float64(551.0)

In [25]:
df['context'].iloc[0]

'At 2133-05-20 23:58:00, the 77-year-old male reports pain level is 10. His vital sign at the time is temperature 36.9°C, heart rate 77 bpm, respiratory rate 14 bpm, O2 saturation 98%, blood pressure 171/66 mmHg. At 2133-05-21 05:15:00, the 77-year-old male reports pain level is NAD. His vital sign at the time is temperature 37.0°C, heart rate 80 bpm, respiratory rate 16 bpm, O2 saturation 99%, blood pressure 100/57 mmHg. At 2133-05-21 09:42:00, the 77-year-old male reports pain level is uta. His vital sign at the time is temperature 36.7°C, heart rate 73 bpm, respiratory rate 25 bpm, O2 saturation 100%, blood pressure 97/52 mmHg. At 2133-05-21 11:36:00, the 77-year-old male reports pain level is uta. His vital sign at the time is temperature 36.6°C, heart rate 65 bpm, respiratory rate 30 bpm, O2 saturation 100%, blood pressure 109/58 mmHg.'

## Templated padding

In [10]:
data_dir = "/Users/vuong/Downloads/research/repos/numeric-reasoning/data/med_numeracy/padding_templated"
df = get_full_data(data_dir)
round(df['context'].map(lambda x: get_num_tokens(x, "gpt-4.1")).mean(), 0)

np.float64(1475.0)

## Variant

In [29]:
data_dir = "/Users/vuong/Downloads/research/repos/numeric-reasoning/data/med_numeracy/natural"
df = get_full_data(data_dir)
round(df['context'].map(lambda x: get_num_tokens(x, "gpt-4.1")).mean(), 0)

np.float64(1480.0)

In [30]:
df['context'].iloc[0]

"At 2133-05-20 23:58:00, A 77-year-old left hand-dominant female first presented to the emergency department (ED) as a transfer from an outlying facility (OLF) secondary to an African grey parrot (Psittacus erithacus) bite. The patient was cleaning her pet parrot’s cage at 09:30 hours when she sustained a bite to the dorsum of the right hand. She experienced immediate pain but upon examination, there were no breaks in the skin. The patient then washed the area with antibacterial soap and applied an ice pack. Throughout the day, the pain, swelling, and bruising continued to increase and eventually spread to her dorsal wrist. The pain became unbearable and so she sought care at her local ED. The OLF performed labs and a hand X-ray. She was given one dose of intravenous (IV) ceftriaxone 1 g and transferred to our ED for evaluation by hand surgery. The patient arrived at our facility and was evaluated at 1931. The patient was complaining of increased pain, numbness, and coolness all over t